## 세포라 top10 수집및 모든리뷰 수집코드

In [1]:
import requests
import json
import time
import random
from datetime import datetime

# ── 설정 ──────────────────────────────────────────────────────────
BV_PASSKEY       = "calXm2DyQVjcCy9agq85vmTJv5ELuuBCF2sdg4BnJzJus"
BV_API_URL       = "https://api.bazaarvoice.com/data/reviews.json"
RANK_SAVE_FILE   = "sephora_rankings_current.jsonl"   
REVIEW_SAVE_FILE = "sephora_reviews_master.jsonl"   

SEPHORA_API_URL = (
    "https://www.sephora.com/api/v2/catalog/categories/skincare/seo"
    "?targetSearchEngine=NLP"
    "&sortBy=P_BEST_SELLING%3A1%3A%3AP_RATING%3A1%3A%3AP_PROD_NAME%3A0"
    "&currentPage=1&pageSize=60&content=true"
    "&includeRegionsMap=true&pickupRampup=true&sddRampup=true"
    "&includeEDD=true&loc=en-US&ch=rwd&user-segment=external-app"
)
HEADERS = {
    "User-Agent"     : "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept"         : "application/json",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer"        : "https://www.sephora.com/shop/skincare?sortBy=BEST_SELLING",
    "Origin"         : "https://www.sephora.com",
}


# ── 함수: 단일 상품 리뷰 전체 수집 ───────────────────────────────
def get_sephora_reviews_all(product_id, product_name):
    all_reviews = []
    limit  = 100
    offset = 0
    total  = None

    print(f"\n  🚀 [{product_name[:35]}] 리뷰 수집 시작...")

    while True:
        params = {
            "Filter"    : ["contentlocale:en*", f"ProductId:{product_id}"],
            "Sort"      : "SubmissionTime:desc",
            "Limit"     : limit,
            "Offset"    : offset,
            "Include"   : "Products,Comments",
            "Stats"     : "Reviews",
            "passkey"   : BV_PASSKEY,
            "apiversion": "5.4",
            "Locale"    : "en_US",
        }
        try:
            response = requests.get(BV_API_URL, params=params, timeout=15)
            data     = response.json()

            if total is None:
                total = data.get("TotalResults", 0)
                print(f"  📊 전체 리뷰 수: {total}개")

            reviews = data.get("Results", [])
            if not reviews:
                print(f"\n  ✅ 수집 완료 (총 {len(all_reviews)}개 / 전체 {total}개)")
                break

            for rev in reviews:
                all_reviews.append({
                    "product_id"    : product_id,
                    "ReviewId"      : rev.get("Id"),
                    "Author"        : rev.get("UserNickname"),
                    "Rating"        : rev.get("Rating"),
                    "Title"         : rev.get("Title"),
                    "ReviewText"    : rev.get("ReviewText"),
                    "SubmissionTime": rev.get("SubmissionTime"),
                    "IsRecommended" : rev.get("IsRecommended"),
                    "Incentivized"  : rev.get("ContextDataValues", {}).get("IncentivizedReview", {}).get("ValueLabel", "N/A"),
                    "Helpfulness"   : rev.get("Helpfulness"),
                })

            offset += limit
            print(f"  🔄 수집 중... {len(all_reviews)} / {total}개", end="\r")
            time.sleep(random.uniform(0.4, 0.7))

            if total and len(all_reviews) >= total:
                print(f"\n  ✅ 수집 완료 (총 {len(all_reviews)}개 / 전체 {total}개)")
                break

        except Exception as e:
            print(f"\n  ❌ 에러 발생: {e}")
            break

    return all_reviews


# ── STEP 1: 세포라 스킨케어 베스트셀러 Top 10 수집 ────────────────
print("📡 Sephora API 호출 중...")
resp = requests.get(SEPHORA_API_URL, headers=HEADERS, timeout=20)
resp.raise_for_status()
data = resp.json()
print(f"✅ 응답 성공! (Status: {resp.status_code})")

products_raw = (
    data.get("products")
    or data.get("catalog", {}).get("products")
    or []
)

if not products_raw:
    print("⚠️ 상품 목록을 찾지 못했습니다. 최상위 키:", list(data.keys()))
    with open("sephora_raw_response.json", "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    print("🔍 sephora_raw_response.json 저장됨 (구조 확인용)")

else:
    rank_data_list    = []
    all_review_master = []   # 전 상품 리뷰 누적
    rank_count        = 1

    for product in products_raw:
        if rank_count > 10:
            break

        # 광고 제외
        if (
            product.get("isSponsored")
            or product.get("sponsored")
            or product.get("adBadge")
            or "sponsored" in str(product.get("badges", [])).lower()
        ):
            continue

        brand       = product.get("brandName", "N/A")
        title       = product.get("displayName", "N/A")
        sku         = product.get("currentSku", {})
        price       = sku.get("listPrice") or sku.get("salePrice") or product.get("listPrice") or "N/A"
        rating      = float(product.get("rating", 0) or 0)
        reviews_cnt = int(product.get("reviews", 0) or 0)
        product_id  = product.get("productId", "N/A")
        url_path    = product.get("targetUrl") or product.get("url", "")
        product_url = ("https://www.sephora.com" + url_path) if url_path and not url_path.startswith("http") else (url_path or "N/A")

        rank_data_list.append({
            "rank"        : rank_count,
            "brand"       : brand,
            "title"       : title,
            "rating"      : rating,
            "reviews"     : reviews_cnt,
            "price"       : price,
            "url"         : product_url,
            "product_id"  : product_id,
            "platform"    : "Sephora",
            "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        })
        print(f"\n📍 {rank_count}위: [{brand}] {title[:40]}")

        # ── STEP 2: 전 상품 리뷰 수집 ─────────────────────────────
        reviews = get_sephora_reviews_all(product_id, title)
        all_review_master.extend(reviews)
        print(f"  → 누적 리뷰: {len(all_review_master)}개")

        rank_count += 1

    # ── STEP 3: JSONL 저장 (Ulta와 동일 형식) ─────────────────────
    with open(RANK_SAVE_FILE, "w", encoding="utf-8") as f:
        for entry in rank_data_list:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

    with open(REVIEW_SAVE_FILE, "w", encoding="utf-8") as f:
        for review in all_review_master:
            f.write(json.dumps(review, ensure_ascii=False) + "\n")

    # ── STEP 4: 결과 출력 ─────────────────────────────────────────
    print()
    print("=" * 65)
    print("🏆 세포라 스킨케어 베스트셀러 Top 10 (광고 제외)")
    print("=" * 65)
    for item in rank_data_list:
        print(f"{item['rank']:>2}위 | {item['brand']:<20} | {item['title'][:28]:<28} | ⭐{item['rating']} | {item['price']}")
    print("=" * 65)
    print(f"\n📊 JSONL 저장 완료")
    print(f"  - 순위 파일 : {RANK_SAVE_FILE} ({len(rank_data_list)}개 상품)")
    print(f"  - 리뷰 파일 : {REVIEW_SAVE_FILE} (총 {len(all_review_master)}개 리뷰)")

📡 Sephora API 호출 중...
✅ 응답 성공! (Status: 200)

📍 1위: [rhode] Glazing Milk Ceramide Facial Essence

  🚀 [Glazing Milk Ceramide Facial Essenc] 리뷰 수집 시작...
  📊 전체 리뷰 수: 1931개
  🔄 수집 중... 1931 / 1931개
  ✅ 수집 완료 (총 1931개 / 전체 1931개)
  → 누적 리뷰: 1931개

📍 2위: [rhode] Peptide Lip Tint Nourishing Glaze

  🚀 [Peptide Lip Tint Nourishing Glaze] 리뷰 수집 시작...
  📊 전체 리뷰 수: 2168개
  🔄 수집 중... 2168 / 2168개
  ✅ 수집 완료 (총 2168개 / 전체 2168개)
  → 누적 리뷰: 4099개

📍 3위: [The Ordinary] Niacinamide 10% + Zinc 1%  Serum for Oil

  🚀 [Niacinamide 10% + Zinc 1%  Serum fo] 리뷰 수집 시작...
  📊 전체 리뷰 수: 8909개
  🔄 수집 중... 8909 / 8909개
  ✅ 수집 완료 (총 8909개 / 전체 8909개)
  → 누적 리뷰: 13008개

📍 4위: [Beauty of Joseon] Day Dew Sunscreen Lightweight SPF 50

  🚀 [Day Dew Sunscreen Lightweight SPF 5] 리뷰 수집 시작...
  📊 전체 리뷰 수: 339개
  🔄 수집 중... 339 / 339개
  ✅ 수집 완료 (총 339개 / 전체 339개)
  → 누적 리뷰: 13347개

📍 5위: [Biodance] Bio Collagen Real Deep Mask for Pore Min

  🚀 [Bio Collagen Real Deep Mask for Por] 리뷰 수집 시작...
  📊 전체 리뷰 수: 473개
  🔄 수집 중... 4

## 팀원이 준 리뷰데이터 번역코드

In [ ]:
"""
translate_pipeline_sephora_full.py
- sephora 원본 전체 리뷰 번역
- 영어 → 한국어
"""

from deep_translator import GoogleTranslator
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
import json, time, re
from tqdm import tqdm

# ── 설정 ─────────────────────────────────────────────────────
INPUT_FILE  = "./data/crawling/sephora_reviews_master.jsonl"   # 원본 전체 파일
OUTPUT_FILE = "sephora_master_translated_en_ko.jsonl"

BODY_COL    = "ReviewText"
ITEM_ID_COL = "product_id"
RATING_COL  = "Rating"

N_SAMPLE    = None   # None이면 전체, 숫자 넣으면 앞 N건만
CHUNK_SIZE  = 5
MAX_WORKERS = 4
MAX_RETRIES = 3
RETRY_SLEEP = 2.0
CHUNK_DELAY = 0.5


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 줄번호 방식 파싱
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def parse_numbered(text: str, expected_n: int) -> list[str]:
    pattern = re.compile(r'\[(\d+)\]\s*(.*?)(?=\[\d+\]|$)', re.DOTALL)
    matches = pattern.findall(text)
    result = {int(idx): body.strip() for idx, body in matches}
    return [result.get(i + 1, "") for i in range(expected_n)]


def build_numbered(texts: list[str]) -> str:
    return "\n".join(f"[{i+1}] {t}" for i, t in enumerate(texts))


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# jsonl 로드
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def load_jsonl(path: str) -> list[dict]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 단건 번역 (fallback)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def translate_single(text: str, src: str, tgt: str) -> str:
    if not text or not str(text).strip():
        return ""

    for attempt in range(MAX_RETRIES):
        try:
            result = GoogleTranslator(source=src, target=tgt).translate(str(text))
            if result and result.strip():
                return result.strip()
        except Exception:
            pass

        time.sleep(RETRY_SLEEP * (attempt + 1))

    return "번역실패"


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 청크 번역 — 영어 → 한국어
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def translate_chunk_en_ko(texts: list[str]) -> list[str]:
    if not texts:
        return []

    joined = build_numbered(texts)

    for attempt in range(MAX_RETRIES):
        try:
            result = GoogleTranslator(source="en", target="ko").translate(joined)
            if not result:
                raise ValueError("빈 응답")

            parts = parse_numbered(result, len(texts))

            if all(p for p in parts):
                return parts

            for i, p in enumerate(parts):
                if not p:
                    parts[i] = translate_single(texts[i], "en", "ko")
            return parts

        except Exception:
            time.sleep(RETRY_SLEEP * (attempt + 1))

    return [translate_single(t, "en", "ko") for t in texts]


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 메인
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def main():
    print(":inbox_tray: 원본 전체 데이터 로드...")
    raw_records = load_jsonl(INPUT_FILE)
    df = pd.DataFrame(raw_records)

    if N_SAMPLE:
        df = df.head(N_SAMPLE)

    print(f"   총 {len(df):,}건 / 컬럼: {list(df.columns)}")

    required_cols = [BODY_COL, ITEM_ID_COL, RATING_COL]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise KeyError(f"필수 컬럼 없음: {missing_cols}")

    texts = df[BODY_COL].fillna("").astype(str).tolist()
    chunks = [texts[i:i+CHUNK_SIZE] for i in range(0, len(texts), CHUNK_SIZE)]
    n_chunks = len(chunks)

    print(f"\n:rocket: 전체 번역 시작 (en→ko)")
    print(f"   {len(texts):,}건 / {n_chunks}개 청크 / workers={MAX_WORKERS}")
    print(f"   CHUNK_SIZE={CHUNK_SIZE} / CHUNK_DELAY={CHUNK_DELAY}s")

    start_time = time.time()

    def run_chunk(chunk):
        ko_texts = translate_chunk_en_ko(chunk)
        time.sleep(CHUNK_DELAY)
        return ko_texts

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        chunk_results = list(
            tqdm(
                executor.map(run_chunk, chunks),
                total=n_chunks,
                desc="번역 (en→ko)",
                unit="chunk"
            )
        )

    elapsed = time.time() - start_time
    print(f"\n:stopwatch: 번역 완료: {elapsed:.1f}초 ({elapsed/60:.1f}분)")

    ko_results = []
    for ko_chunk in chunk_results:
        ko_results.extend(ko_chunk)

    df["comment_ko"] = ko_results[:len(df)]

    def success_mask(col):
        return df[col].astype(str).str.strip().ne("") & df[col].ne("번역실패")

    ko_ok = success_mask("comment_ko")

    print(f"\n:bar_chart: 번역 결과")
    print(f"   성공률: {ko_ok.mean()*100:.1f}% ({ko_ok.sum():,}/{len(df):,}건)")
    print(f"   처리 속도: {len(df)/elapsed:.1f}건/초")

    print(f"\n:floppy_disk: 저장 중: {OUTPUT_FILE}")
    output_records = json.loads(df.to_json(orient="records", force_ascii=False))
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        for record in output_records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    print(f":white_check_mark: 저장 완료: {OUTPUT_FILE}")

    print("\n:clipboard: 샘플 3건:")
    for _, row in df.head(3).iterrows():
        print(f"   product_id : {row.get(ITEM_ID_COL, 'N/A')}")
        print(f"   rating     : ★{row.get(RATING_COL, 'N/A')}")
        print(f"   원문(en)   : {str(row[BODY_COL])[:60]}...")
        print(f"   한국(ko)   : {str(row['comment_ko'])[:60]}...")
        print("   " + "─" * 52)


if __name__ == "__main__":
    main()